# SAURIA + Gather: DRAM → SRAM A

Questo notebook genera e simula il nuovo flusso integrato:

```text
dense values + CSR/CSC metadata in external DRAM
                  |
                  v
               Gather
                  |
                  v
          SAURIA SRAM A / fmap
                  |
                  v
          SAURIA computation
```

La sequenza del test è:

1. genera una workload SAURIA **vanilla** a singolo tile;
2. salva una copia degli stimuli vanilla;
3. aggiunge in DRAM i dati di staging e i metadata CSR;
4. inserisce gli accessi AXI-Lite del gather prima dello start del controller;
5. imposta `KEEP_A=1`, così il DMA non sovrascrive la fmap prodotta dal gather;
6. configura il gather con output locale `SRAMA_OFFSET = 0x0004_0000`;
7. esegue Verilator e verifica il risultato SAURIA finale.

> **Campo di validità.** Il notebook usa deliberatamente una workload a singolo tile.
> Per una convoluzione multi-tile il gather deve essere avviato prima di ogni tile A,
> oppure il controller deve essere esteso con una sequenza gather/DMA per-tile.


## 0. Configurazione

Eseguire il notebook dalla repository `sauria-gather`, preferibilmente dalla cartella
`Python/notebooks`. Il kernel deve avere le dipendenze installate tramite `source setup.sh`.


In [1]:
from __future__ import annotations

from pathlib import Path
from math import ceil, log2
import json
import os
import re
import shutil
import subprocess
import sys

import dotenv
import numpy as np
import pandas as pd
import torch
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Find a SAURIA repository without depending on the launch directory."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "RTL").is_dir()
            and (candidate / "Python").is_dir()
            and (candidate / "test").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside sauria-gather."
    )


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
PYTHON_DIR = REPO_ROOT / "Python"
TEST_DIR = REPO_ROOT / "test"
VERILATOR_DIR = TEST_DIR / "verilator"
STIMULI_DIR = TEST_DIR / "stimuli"
VANILLA_STIMULI_DIR = TEST_DIR / "stimuli_vanilla_for_gather_sram"
INTEGRATED_STIMULI_DIR = STIMULI_DIR

for env_file in (PYTHON_DIR / "env", REPO_ROOT / "env"):
    if env_file.exists():
        dotenv.load_dotenv(env_file, override=True)

sys.path.insert(0, str(PYTHON_DIR))
import src.hw_versions as hwv
import src.sauria_lib as slib

SAURIA_VERSION = "FP16_8x16_AXI64"

RUN_COMPILE = True
RUN_VANILLA_REFERENCE = True
RUN_INTEGRATED_SIMULATION = True
ADD_GATHER_PERF_READS = True

MAX_SIM_CYCLES = 2_000_000
RANDOM_SEED = 7

print("REPO_ROOT:", REPO_ROOT)
print("VERILATOR_DIR:", VERILATOR_DIR)
print("STIMULI_DIR:", STIMULI_DIR)


REPO_ROOT: /home/henry/sauria
VERILATOR_DIR: /home/henry/sauria/test/verilator
STIMULI_DIR: /home/henry/sauria/test/stimuli


## 1. Verifica preliminare dell'RTL

Questa cella controlla che il mux con priorità al gather sia incluso nel filelist e che
il subsystem contenga il percorso `gather_sauria_mem → SRAM locali`.


In [2]:
RTL_DIR = REPO_ROOT / "RTL"
GATHER_RTL_DIR = RTL_DIR / "src" / "Gather"

mux_path = GATHER_RTL_DIR / "axi_busy_priority_mux.sv"
subsystem_path = RTL_DIR / "src" / "sauria_subsystem.sv"
frontend_path = GATHER_RTL_DIR / "gather_frontend_axi.sv"
filelist_path = RTL_DIR / "filelist.f"

required_files = [mux_path, subsystem_path, frontend_path, filelist_path]
missing = [str(path) for path in required_files if not path.exists()]
assert not missing, "Missing RTL files:\n" + "\n".join(missing)

filelist_text = filelist_path.read_text(errors="replace")
subsystem_text = subsystem_path.read_text(errors="replace")

assert "axi_busy_priority_mux.sv" in filelist_text, (
    "Add RTL/src/Gather/axi_busy_priority_mux.sv to RTL/filelist.f"
)
assert "gather_sauria_mem" in subsystem_text
assert "axi_busy_priority_mux" in subsystem_text
assert "gather_busy" in subsystem_text

print("RTL integration checks passed.")


RTL integration checks passed.


## 2. Compilazione Verilator

La compilazione usa la stessa versione hardware scelta dalla libreria Python.
Il log completo viene salvato in `test/verilator/verilator_compile_gather_sram.log`.


In [3]:
compile_log = VERILATOR_DIR / "verilator_compile_gather_sram.log"

if RUN_COMPILE:
    cmd = ["sh", "./compile_sauria.sh", SAURIA_VERSION]
    result = subprocess.run(
        cmd,
        cwd=VERILATOR_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    compile_log.write_text(result.stdout)
    print("\n".join(result.stdout.splitlines()[-80:]))
    if result.returncode != 0:
        raise RuntimeError(
            f"Verilator compilation failed with code {result.returncode}. "
            f"See {compile_log}"
        )
    print("Compilation completed:", compile_log)
else:
    print("Compilation skipped. Existing Test-Sim will be used.")


ccache g++  -I.  -MMD -I/home/henry/sauria/tools/verilator/include -I/home/henry/sauria/tools/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -std=c++11 -DEXACT -O2  -std=gnu++14 -Os -c -o Vsauria_tester_sa_processing_element__pi60__DepSet_hc8156634__0.o Vsauria_tester_sa_processing_element__pi60__DepSet_hc8156634__0.cpp
ccache g++  -I.  -MMD -I/home/henry/sauria/tools/verilator/include -I/home/henry/sauria/tools/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -std=c++11 -DEXACT -O2  -std=gnu++14 -Os -c -o Vsauria_tester_sa_processing

## 3. Workload minima a singolo tile

Il caso scelto usa:

- input: `16 × 1 × 8`;
- pesi: `16 × 16 × 1 × 1`;
- output: `16 × 1 × 8`;
- valori fmap caricati dal gather: `128 FP16`.

`128` indici rientrano nella SRAM indici corrente (`128` linee AXI64, due indici
a 32 bit per linea) e i dati densi rientrano in un blocco del gather.


In [4]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

C_in = 16
C_out = 16
Kh, Kw = 1, 1
stride = 1
dilation = 1

Cw = 8
Ch = 1
Aw = (1 + stride * (Cw - 1)) + (1 + dilation * (Kw - 1)) - 1
Ah = (1 + stride * (Ch - 1)) + (1 + dilation * (Kh - 1)) - 1

with torch.no_grad():
    conv = torch.nn.Conv2d(
        C_in,
        C_out,
        (Kh, Kw),
        stride=stride,
        dilation=dilation,
        bias=False,
    )
    tensor_A_torch = torch.randn(C_in, Ah, Aw, dtype=torch.float32)
    tensor_C_torch = conv(tensor_A_torch)

tensor_A = tensor_A_torch.detach().cpu().numpy().copy()
tensor_B = conv.weight.detach().cpu().numpy().copy()
tensor_C = tensor_C_torch.detach().cpu().numpy().copy()
preload_C = np.zeros_like(tensor_C, dtype=np.float32)

HW_PARAMS = hwv.get_params(SAURIA_VERSION)

TILING_DICT = {
    "C_tile_shape": [16, 1, 8],
    "tile_cin": 16,
    "X_used": 16,
    "Y_used": 8,
}

CONV_DICT = slib.get_conv_dict(
    [tensor_A.shape, tensor_B.shape, tensor_C.shape],
    TILING_DICT,
    HW_PARAMS,
    d=dilation,
    s=stride,
    preloads=False,
)

N_VALUES = (
    CONV_DICT["A_w_til"]
    * CONV_DICT["A_h_til"]
    * CONV_DICT["c_til"]
)

GATHER_DENSE_CAPACITY_FP16 = 128 * (64 // 16)
GATHER_INDEX_CAPACITY = 128 * (64 // 32)

assert CONV_DICT["N_total_tiles"] == 1, (
    "This notebook is intentionally restricted to one external SAURIA tile."
)
assert N_VALUES <= GATHER_DENSE_CAPACITY_FP16
assert N_VALUES <= GATHER_INDEX_CAPACITY

print("A shape:", tensor_A.shape)
print("B shape:", tensor_B.shape)
print("C shape:", tensor_C.shape)
print("N_total_tiles:", CONV_DICT["N_total_tiles"])
print("Gathered FP16 values:", N_VALUES)
print("Index capacity:", GATHER_INDEX_CAPACITY)


A shape: (16, 1, 8)
B shape: (16, 16, 1, 1)
C shape: (16, 1, 8)
N_total_tiles: 1
Gathered FP16 values: 128
Index capacity: 256


## 4. Generazione degli stimuli vanilla

La libreria SAURIA genera e verifica prima il test standard. Gli stimuli risultanti
vengono copiati in una directory separata e usati come base per il test integrato.


In [5]:
def copy_stimuli_set(src: Path, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    for name in ("GoldenStimuli.txt", "initial_dram.txt", "gold_dram.txt", "tstcfg.txt"):
        source = src / name
        if not source.exists():
            matches = sorted(src.glob(name.replace(".txt", "*.txt")))
            if not matches:
                raise FileNotFoundError(source)
            source = matches[0]
        shutil.copy2(source, dst / name)


if RUN_VANILLA_REFERENCE:
    SAURIA_outputs_vanilla, SAURIA_stats_vanilla = slib.Conv2d_SAURIA(
        tensor_A,
        tensor_B,
        preload_C,
        tensor_C,
        CONV_DICT,
        HW_PARAMS,
        generate_vcd=False,
        assert_no_errors=True,
        print_statistics=True,
        test_dir=str(TEST_DIR),
        silent=False,
    )
    copy_stimuli_set(STIMULI_DIR, VANILLA_STIMULI_DIR)
    print("Vanilla stimuli copied to:", VANILLA_STIMULI_DIR)
else:
    copy_stimuli_set(VANILLA_STIMULI_DIR, STIMULI_DIR)
    print("Using existing vanilla stimuli from:", VANILLA_STIMULI_DIR)



              TEST PASSED

****************************************
          SAURIA STATISTICS
****************************************
Total cycles:				637
Total operations:			4096
Average Throughput:			6.43 OP/cycle (2.51 %)

Number of tiles:			1
Core stall cycles:			102 (16.01 %)
Memory/CGF stall cycles:		515 (80.85 %)

SAURIA memory capacity (A|B|C):		32.0 | 32.0 | 32.0 [kB]
Utilized memory:			0.25 | 0.5 | 0.25 [kB] (0.78 | 1.56 | 0.78 [%])
Vanilla stimuli copied to: /home/henry/sauria/test/stimuli_vanilla_for_gather_sram


## 5. Helper per trasformare gli stimuli

Le funzioni seguenti:

- copiano il tile A originale in una zona di staging DRAM;
- costruiscono metadata CSR identity (`col_idx = 0,1,...,N-1`);
- azzerano tutto il padding AXI dei metadata;
- aggiungono un beat indice nullo extra, compatibile con il comportamento corrente
  della FSM del gather;
- impostano il bit `KEEP_A`;
- inseriscono la configurazione gather subito prima dello start SAURIA;
- scrivono il risultato del gather a `0x0004_0000`, cioè SRAM A locale.


In [6]:
GATHER_BASE = 0x7000_0000
CTRL_START_ADDR = 0x4000_0000
CTRL_FLAGS_ADDR = 0x4000_0064
CTRL_KEEP_A_BIT = 19

SAURIA_BASE = 0x5000_0000
SRAMA_LOCAL = 0x0004_0000
SRAMA_DEBUG = SAURIA_BASE + SRAMA_LOCAL

REG_START = 0x00
REG_IS_SPMM = 0x04
REG_DONE = 0x08
REG_AR_SIZE = 0x0C
REG_AW_SIZE = 0x10
REG_AR_ADDR_DENSE_MATRIX = 0x14
REG_TOTAL_LEN_DENSE_MATRIX = 0x18
REG_AR_ADDR_COMP_IDX = 0x1C
REG_TOTAL_LEN_COMP_IDX = 0x20
REG_AR_ADDR_IDX = 0x24
REG_TOTAL_LEN_IDX = 0x28
REG_AXI_WR_AW_ADDR_IN = 0x2C
REG_STATUS_REG_N_ROW = 0x30

PERF_REGS = {
    GATHER_BASE + 0x34: "gather_cycles",
    GATHER_BASE + 0x38: "rd_cycles",
    GATHER_BASE + 0x3C: "wr_cycles",
    GATHER_BASE + 0x40: "r_stall",
    GATHER_BASE + 0x44: "w_stall",
    GATHER_BASE + 0x48: "r_beats",
    GATHER_BASE + 0x4C: "w_beats",
}

AXI_BYTES = 8
ELEM_BYTES = 2
IDX_BYTES = 4

NUM_WORDS = 128
N_BLOCKS = 8
WORDS_PER_LINE = AXI_BYTES // ELEM_BYTES
BYTE_SEL_BITS = int(log2(WORDS_PER_LINE))
ADDR_BITS = int(log2(NUM_WORDS))


def gaddr(offset: int) -> int:
    return GATHER_BASE + offset


def nbeats(nbytes: int) -> int:
    return ceil(nbytes / AXI_BYTES)


def align_up(value: int, alignment: int = 0x1000) -> int:
    return (value + alignment - 1) & ~(alignment - 1)


def stim_write(addr: int, data: int) -> list[int]:
    return [data & 0xFFFFFFFF, addr & 0xFFFFFFFF, 1, 0, 0, 0, 0]


def stim_read(addr: int, expected: int = 0, final_check: bool = False) -> list[int]:
    return [0, addr & 0xFFFFFFFF, 0, 1, 0, expected & 0xFFFFFFFF, int(final_check)]


def stim_wait_gather() -> list[int]:
    return [0, 0, 0, 0, 2, 0, 0]


def read_stimuli(path: Path) -> list[list[int]]:
    rows = []
    for lineno, line in enumerate(path.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        values = [int(token, 16) for token in line.split()]
        if len(values) != 7:
            raise ValueError(f"Bad row at {path}:{lineno}: {line}")
        rows.append(values)
    return rows


def write_stimuli(path: Path, rows: list[list[int]]) -> None:
    with path.open("w") as stream:
        for row in rows:
            stream.write(" ".join(f"{int(value) & 0xFFFFFFFF:X}" for value in row) + "\n")


def read_byte_mem(path: Path) -> list[int]:
    return [int(line.strip(), 16) & 0xFF for line in path.read_text().splitlines() if line.strip()]


def write_byte_mem(path: Path, memory: list[int]) -> None:
    with path.open("w") as stream:
        for byte in memory:
            stream.write(f"{int(byte) & 0xFF:X}\n")


def ensure_len(memory: list[int], size: int) -> None:
    if len(memory) < size:
        memory.extend([0] * (size - len(memory)))


def read_bytes(memory: list[int], addr: int, size: int) -> list[int]:
    if addr < 0 or size < 0:
        raise ValueError("Negative memory range")
    ensure_len(memory, addr + size)
    return list(memory[addr:addr + size])


def write_bytes(memory: list[int], addr: int, data: list[int]) -> None:
    ensure_len(memory, addr + len(data))
    memory[addr:addr + len(data)] = [int(value) & 0xFF for value in data]


def zero_range(memory: list[int], addr: int, size: int) -> None:
    write_bytes(memory, addr, [0] * size)


def write_le(memory: list[int], addr: int, value: int, size: int) -> None:
    ensure_len(memory, addr + size)
    for byte_idx in range(size):
        memory[addr + byte_idx] = (value >> (8 * byte_idx)) & 0xFF


def pack_dense_index(linear_index: int, block: int = 0) -> int:
    byte_sel = linear_index % WORDS_PER_LINE
    word_addr = linear_index // WORDS_PER_LINE
    if word_addr >= NUM_WORDS:
        raise ValueError(f"linear_index={linear_index} exceeds one dense gather block")
    if block >= N_BLOCKS:
        raise ValueError(f"block={block} exceeds N_BLOCKS={N_BLOCKS}")
    return ((block << (ADDR_BITS + BYTE_SEL_BITS))
            | (word_addr << BYTE_SEL_BITS)
            | byte_sel)


def patch_identity_csr(memory: list[int], *, comp_base: int, idx_base: int,
                       n_values: int, idx_extra_beats: int) -> dict[str, int]:
    comp_beats = nbeats(3 * IDX_BYTES)
    idx_natural_beats = nbeats(n_values * IDX_BYTES)
    idx_total_beats = idx_natural_beats + idx_extra_beats

    zero_range(memory, comp_base, comp_beats * AXI_BYTES)
    zero_range(memory, idx_base, idx_total_beats * AXI_BYTES)

    for index, value in enumerate((0, n_values, 0)):
        write_le(memory, comp_base + index * IDX_BYTES, value, IDX_BYTES)
    for index in range(n_values):
        write_le(memory, idx_base + index * IDX_BYTES,
                 pack_dense_index(index), IDX_BYTES)

    return {
        "comp_beats": comp_beats,
        "idx_natural_beats": idx_natural_beats,
        "idx_total_beats": idx_total_beats,
    }


def set_keep_a(rows: list[list[int]]) -> int:
    patched = 0
    for row in rows:
        data, addr, write_enable = row[0], row[1], row[2]
        if write_enable and addr == CTRL_FLAGS_ADDR:
            row[0] = data | (1 << CTRL_KEEP_A_BIT)
            patched += 1
    if patched == 0:
        raise RuntimeError(f"Controller flags write 0x{CTRL_FLAGS_ADDR:08X} not found")
    return patched


def find_controller_start(rows: list[list[int]]) -> int:
    for index, row in enumerate(rows):
        data, addr, write_enable = row[0], row[1], row[2]
        if write_enable and addr == CTRL_START_ADDR and data == 3:
            return index
    raise RuntimeError("Controller start row not found: data=3, addr=0x40000000")


def build_gather_rows(*, dense_base: int, comp_base: int, idx_base: int,
                      srama_addr: int, n_values: int, n_dense_cols: int,
                      idx_extra_beats: int, add_perf_reads: bool) -> list[list[int]]:
    comp_beats = nbeats(3 * IDX_BYTES)
    idx_beats = nbeats(n_values * IDX_BYTES) + idx_extra_beats

    rows = [
        stim_write(gaddr(REG_AR_SIZE), 3),
        stim_write(gaddr(REG_AW_SIZE), 3),
        stim_write(gaddr(REG_AR_ADDR_DENSE_MATRIX), dense_base),
        stim_write(gaddr(REG_TOTAL_LEN_DENSE_MATRIX), nbeats(n_dense_cols * ELEM_BYTES)),
        stim_write(gaddr(REG_AR_ADDR_COMP_IDX), comp_base),
        stim_write(gaddr(REG_TOTAL_LEN_COMP_IDX), comp_beats),
        stim_write(gaddr(REG_AR_ADDR_IDX), idx_base),
        stim_write(gaddr(REG_TOTAL_LEN_IDX), idx_beats),
        stim_write(gaddr(REG_AXI_WR_AW_ADDR_IN), srama_addr),
        stim_write(gaddr(REG_STATUS_REG_N_ROW), n_dense_cols - 1),
        stim_write(gaddr(REG_IS_SPMM), 0),
        stim_write(gaddr(REG_START), 1),
        stim_wait_gather(),
        stim_read(gaddr(REG_DONE), expected=1, final_check=False),
    ]
    if add_perf_reads:
        rows.extend(stim_read(addr, expected=0, final_check=False) for addr in PERF_REGS)
    return rows


def make_gather_to_srama_stimuli(*, in_dir: Path, out_dir: Path,
                                  n_values: int, n_dense_cols: int,
                                  ifmap_dram_base: int = 0,
                                  dense_stage_base: int = 0x0009_0000,
                                  comp_base: int | None = None,
                                  idx_base: int | None = None,
                                  srama_offset_bytes: int = 0,
                                  idx_extra_beats: int = 1,
                                  add_perf_reads: bool = True) -> dict:
    if n_values <= 0 or n_dense_cols <= 0:
        raise ValueError("n_values and n_dense_cols must be positive")
    if n_values > n_dense_cols:
        raise ValueError("Identity CSR requires n_values <= n_dense_cols")
    if n_values > GATHER_INDEX_CAPACITY:
        raise ValueError(f"{n_values} indices exceed capacity {GATHER_INDEX_CAPACITY}")
    if n_dense_cols > GATHER_DENSE_CAPACITY_FP16:
        raise ValueError(f"{n_dense_cols} dense values exceed capacity {GATHER_DENSE_CAPACITY_FP16}")
    if srama_offset_bytes % AXI_BYTES:
        raise ValueError("SRAM A output offset must be AXI64 aligned")

    in_dir = Path(in_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = read_stimuli(in_dir / "GoldenStimuli.txt")
    initial_dram = read_byte_mem(in_dir / "initial_dram.txt")
    gold_dram = read_byte_mem(in_dir / "gold_dram.txt")

    dense_nbytes = n_dense_cols * ELEM_BYTES
    gathered_nbytes = n_values * ELEM_BYTES
    dense_source = read_bytes(initial_dram, ifmap_dram_base, dense_nbytes)

    if comp_base is None:
        comp_base = align_up(dense_stage_base + dense_nbytes)
    comp_beats = nbeats(3 * IDX_BYTES)
    if idx_base is None:
        idx_base = align_up(comp_base + comp_beats * AXI_BYTES)
    idx_total_beats = nbeats(n_values * IDX_BYTES) + idx_extra_beats

    ranges = {
        "dense_stage": (dense_stage_base, dense_stage_base + dense_nbytes),
        "comp_idx": (comp_base, comp_base + comp_beats * AXI_BYTES),
        "idx": (idx_base, idx_base + idx_total_beats * AXI_BYTES),
    }
    names = list(ranges)
    for left_idx, left_name in enumerate(names):
        left_start, left_end = ranges[left_name]
        for right_name in names[left_idx + 1:]:
            right_start, right_end = ranges[right_name]
            if left_start < right_end and right_start < left_end:
                raise ValueError(f"DRAM staging overlap: {left_name} and {right_name}")

    for memory in (initial_dram, gold_dram):
        write_bytes(memory, dense_stage_base, dense_source)
        metadata = patch_identity_csr(
            memory,
            comp_base=comp_base,
            idx_base=idx_base,
            n_values=n_values,
            idx_extra_beats=idx_extra_beats,
        )

    patched_flag_rows = set_keep_a(rows)
    start_index = find_controller_start(rows)
    srama_addr = SRAMA_LOCAL + srama_offset_bytes
    gather_rows = build_gather_rows(
        dense_base=dense_stage_base,
        comp_base=comp_base,
        idx_base=idx_base,
        srama_addr=srama_addr,
        n_values=n_values,
        n_dense_cols=n_dense_cols,
        idx_extra_beats=idx_extra_beats,
        add_perf_reads=add_perf_reads,
    )
    rows = rows[:start_index] + gather_rows + rows[start_index:]

    write_stimuli(out_dir / "GoldenStimuli.txt", rows)
    write_byte_mem(out_dir / "initial_dram.txt", initial_dram)
    write_byte_mem(out_dir / "gold_dram.txt", gold_dram)
    shutil.copy2(in_dir / "tstcfg.txt", out_dir / "tstcfg.txt")

    manifest = {
        "mode": "gather_dram_read_to_srama_then_sauria_keep_a",
        "n_values": n_values,
        "n_dense_cols": n_dense_cols,
        "gathered_nbytes": gathered_nbytes,
        "ifmap_dram_base": f"0x{ifmap_dram_base:08X}",
        "dense_stage_base": f"0x{dense_stage_base:08X}",
        "comp_base": f"0x{comp_base:08X}",
        "idx_base": f"0x{idx_base:08X}",
        "srama_addr": f"0x{srama_addr:08X}",
        "idx_extra_beats": idx_extra_beats,
        "metadata": metadata,
        "keep_a_flag_rows_patched": patched_flag_rows,
        "gather_rows_inserted": len(gather_rows),
        "perf_reads_enabled": add_perf_reads,
        "input_dir": str(in_dir),
        "output_dir": str(out_dir),
    }
    (out_dir / "gather_srama_manifest.json").write_text(json.dumps(manifest, indent=2))
    return manifest


## 6. Generazione degli stimuli integrati

Il tile A vanilla viene letto da DRAM base `0x0`, copiato nella zona di staging e
gathered con una mappa identity. Il risultato viene scritto direttamente in SRAM A.


In [7]:
IFMAP_DRAM_BASE = 0x0000_0000
DENSE_STAGE_BASE = 0x0009_0000
SRAMA_OUTPUT_OFFSET = 0
IDX_EXTRA_BEATS = 1

manifest = make_gather_to_srama_stimuli(
    in_dir=VANILLA_STIMULI_DIR,
    out_dir=INTEGRATED_STIMULI_DIR,
    n_values=N_VALUES,
    n_dense_cols=N_VALUES,
    ifmap_dram_base=IFMAP_DRAM_BASE,
    dense_stage_base=DENSE_STAGE_BASE,
    srama_offset_bytes=SRAMA_OUTPUT_OFFSET,
    idx_extra_beats=IDX_EXTRA_BEATS,
    add_perf_reads=ADD_GATHER_PERF_READS,
)
print(json.dumps(manifest, indent=2))


{
  "mode": "gather_dram_read_to_srama_then_sauria_keep_a",
  "n_values": 128,
  "n_dense_cols": 128,
  "gathered_nbytes": 256,
  "ifmap_dram_base": "0x00000000",
  "dense_stage_base": "0x00090000",
  "comp_base": "0x00091000",
  "idx_base": "0x00092000",
  "srama_addr": "0x00040000",
  "idx_extra_beats": 1,
  "metadata": {
    "comp_beats": 2,
    "idx_natural_beats": 64,
    "idx_total_beats": 65
  },
  "keep_a_flag_rows_patched": 1,
  "gather_rows_inserted": 21,
  "perf_reads_enabled": true,
  "input_dir": "/home/henry/sauria/test/stimuli_vanilla_for_gather_sram",
  "output_dir": "/home/henry/sauria/test/stimuli"
}


## 7. Controlli statici sugli stimuli

Prima della simulazione verifichiamo `KEEP_A`, l'indirizzo SRAM A, il wait del gather,
l'ordine rispetto allo start SAURIA e la coerenza delle zone aggiunte alla DRAM.


In [8]:
patched_rows = read_stimuli(INTEGRATED_STIMULI_DIR / "GoldenStimuli.txt")
controller_start_index = find_controller_start(patched_rows)

flag_writes = [row for row in patched_rows if row[2] and row[1] == CTRL_FLAGS_ADDR]
assert flag_writes
assert all(row[0] & (1 << CTRL_KEEP_A_BIT) for row in flag_writes)

gather_output_writes = [
    row for row in patched_rows[:controller_start_index]
    if row[2] and row[1] == gaddr(REG_AXI_WR_AW_ADDR_IN)
]
assert gather_output_writes
assert gather_output_writes[-1][0] == SRAMA_LOCAL + SRAMA_OUTPUT_OFFSET

wait_rows = [row for row in patched_rows[:controller_start_index] if row[4] == 2]
assert wait_rows, "Gather wait row was not inserted"

initial_patched = read_byte_mem(INTEGRATED_STIMULI_DIR / "initial_dram.txt")
gold_patched = read_byte_mem(INTEGRATED_STIMULI_DIR / "gold_dram.txt")

dense_stage_base_int = int(manifest["dense_stage_base"], 16)
comp_base_int = int(manifest["comp_base"], 16)
idx_base_int = int(manifest["idx_base"], 16)

dense_bytes = N_VALUES * ELEM_BYTES
comp_bytes = manifest["metadata"]["comp_beats"] * AXI_BYTES
idx_bytes = manifest["metadata"]["idx_total_beats"] * AXI_BYTES

for start, size in (
    (dense_stage_base_int, dense_bytes),
    (comp_base_int, comp_bytes),
    (idx_base_int, idx_bytes),
):
    assert initial_patched[start:start + size] == gold_patched[start:start + size]

gather_cfg = []
for row in patched_rows[:controller_start_index]:
    if row[2] and GATHER_BASE <= row[1] < GATHER_BASE + 0x100:
        gather_cfg.append({"address": f"0x{row[1]:08X}", "data": f"0x{row[0]:08X}"})

display(pd.DataFrame(gather_cfg))
print("Static stimulus checks passed.")


,address,data
0,0x7000000C,0x00000003
1,0x70000010,0x00000003
2,0x70000014,0x00090000
3,0x70000018,0x00000020
4,0x7000001C,0x00091000
5,0x70000020,0x00000002
6,0x70000024,0x00092000
7,0x70000028,0x00000041
8,0x7000002C,0x00040000
9,0x70000030,0x0000007F


Static stimulus checks passed.


## 8. Simulazione integrata

La simulazione deve mostrare il completamento del gather e terminare con `SUCCESS!`.
Il log completo viene conservato in `test/verilator/sauria_gather_srama.log`.


In [9]:
simulation_log = VERILATOR_DIR / "sauria_gather_srama.log"

if RUN_INTEGRATED_SIMULATION:
    test_sim = VERILATOR_DIR / "Test-Sim"
    assert test_sim.exists(), f"{test_sim} not found. Run the compilation cell first."

    cmd = [str(test_sim), "+debug", f"+max-cycles={MAX_SIM_CYCLES}"]
    print("Running:", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=VERILATOR_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    simulation_log.write_text(result.stdout)
    print("\n".join(result.stdout.splitlines()[-180:]))
    print("returncode:", result.returncode)
    print("full log:", simulation_log)

    output_lower = result.stdout.lower()
    if result.returncode != 0:
        raise RuntimeError(f"Integrated simulation failed with code {result.returncode}")
    assert "gather done" in output_lower, "The testbench did not report Gather completion."
    assert "success!" in output_lower, "The integrated result did not pass the golden check."

    SIMULATION_OUTPUT = result.stdout
    print("SAURIA + Gather SRAM-A test passed.")
else:
    SIMULATION_OUTPUT = simulation_log.read_text(errors="replace") if simulation_log.exists() else ""
    print("Simulation skipped.")


Running: /home/henry/sauria/test/verilator/Test-Sim +debug +max-cycles=2000000
[760] CFG idx 31
[770] CFG idx 32
Writing C08040 into address 4000008C
[780] CFG idx 32
[790] CFG idx 33
Writing 1C18141 into address 40000090
[800] CFG idx 33
[810] CFG idx 34
Writing 40080100 into address 40000094
[820] CFG idx 34
[830] CFG idx 35
Writing 20000 into address 40000098
[840] CFG idx 35
[850] CFG idx 36
Writing FC008001 into address 4000009C
[860] CFG idx 36
[870] CFG idx 37
Writing 7FF into address 400000A0
[880] CFG idx 37
[890] CFG idx 38
Writing 80001 into address 400000A4
[900] CFG idx 38
[910] CFG idx 39
Writing 80100002 into address 400000A8
[920] CFG idx 39
[930] CFG idx 40
Writing 20004000 into address 400000AC
[940] CFG idx 40
[950] CFG idx 41
Writing 80010000 into address 400000B0
[960] CFG idx 41
[970] CFG idx 42
Writing 0 into address 400000B4
[980] CFG idx 42
[990] CFG idx 43
Writing 3 into address 7000000C
[1000] CFG idx 43
[1010] CFG idx 44
Writing 3 into address 70000010
[1020

## 9. Performance del gather

Se i registri performance sono esposti dal frontend, questa cella estrae cicli, stall,
beat AXI e bandwidth effettiva dal log Verilator.


In [10]:
READ_RE = re.compile(r"Read\\s+([0-9A-Fa-f]+)\\s+from address\\s+([0-9A-Fa-f]+)")

perf_values = {}
for match in READ_RE.finditer(SIMULATION_OUTPUT):
    value = int(match.group(1), 16)
    address = int(match.group(2), 16)
    if address in PERF_REGS:
        perf_values[PERF_REGS[address]] = value

if perf_values:
    ordered_names = list(PERF_REGS.values())
    display(pd.DataFrame([
        {"metric": name, "value": perf_values.get(name)}
        for name in ordered_names
    ]))

    total_cycles = perf_values.get("gather_cycles", 0)
    r_beats = perf_values.get("r_beats", 0)
    w_beats = perf_values.get("w_beats", 0)
    summary = {
        "read_bytes": r_beats * AXI_BYTES,
        "write_bytes": w_beats * AXI_BYTES,
        "read_bytes_per_cycle": (r_beats * AXI_BYTES) / total_cycles if total_cycles else None,
        "write_bytes_per_cycle": (w_beats * AXI_BYTES) / total_cycles if total_cycles else None,
    }
    display(pd.DataFrame([summary]))
else:
    print("No performance-register reads found. Enable ADD_GATHER_PERF_READS and registers 0x34..0x4C.")


No performance-register reads found. Enable ADD_GATHER_PERF_READS and registers 0x34..0x4C.


## 10. Interpretazione del risultato

Un test riuscito dimostra congiuntamente che:

1. il gather legge dense values e metadata dalla DRAM;
2. il mux assegna al gather l'accesso alle SRAM locali durante `gather_busy`;
3. i burst `AW/W/B` raggiungono SRAM A;
4. `KEEP_A` impedisce al DMA di sovrascrivere la fmap;
5. SAURIA usa la fmap prodotta dal gather e genera il risultato golden.

Il confronto finale della DRAM non controlla direttamente SRAM A, ma la correttezza
dell'output SAURIA rende il controllo end-to-end: dati errati o non scritti nella fmap
producono normalmente un mismatch nel risultato finale.
